# Fine-tuning Qwen3.5-4B di Google Colab

Notebook ini melatih adapter **LoRA** untuk [`Qwen/Qwen3.5-4B`](https://huggingface.co/Qwen/Qwen3.5-4B) (Apache 2.0)
dengan dataset di repo ini, memakai [Unsloth](https://github.com/unslothai/unsloth). Model dilatih dalam
**mode non-thinking**: model langsung menjawab tanpa blok `<think>`, sesuai format dataset.

**Sebelum mulai:** menu *Runtime → Change runtime type*, pilih GPU, lalu jalankan sel dari atas ke bawah.

| GPU | Presisi | `own` (62 sampel, 3 epoch) | `mix` (±960 sampel, 2 epoch) |
|---|---|---|---|
| T4 (gratis) | float32* | ±30–90 menit | ±2–5 jam |
| L4 (Colab berbayar) | bf16 | ±10–20 menit | ±40–80 menit |

\* T4 tidak mendukung bf16, dan Qwen3.5 menghasilkan NaN pada fp16. Karena itu Unsloth otomatis melatih dalam
float32 di T4, yang lebih lambat. Waktu di atas hanya perkiraan. Sel *Siapkan data* menghitung token yang
sebenarnya, dan progress bar training menampilkan sisa waktu (ETA) yang akurat.

Checkpoint disimpan ke Google Drive setiap beberapa step. Jika sesi Colab putus, jalankan ulang semua sel dan
training otomatis **dilanjutkan dari checkpoint terakhir**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Instal Unsloth
Butuh ±2–4 menit. Qwen3.5 membutuhkan transformers v5, yang ikut terpasang bersama Unsloth versi terbaru.
Jika Colab meminta *restart session* setelah instalasi, lakukan lalu lanjutkan ke sel berikutnya.

In [ ]:
%%capture
!pip install --upgrade unsloth

## 2. Konfigurasi
Semua pengaturan ada di sel ini. `MODEL = "qwen2.5-coder-7b"` melatih model lama sebagai pembanding dengan data dan
prompt uji yang sama, dan hasilnya disimpan di folder Drive terpisah.

In [ ]:
MODEL         = "qwen3.5-4b"    # "qwen3.5-4b" atau "qwen2.5-coder-7b" (pembanding)
DATASET       = "own"           # "own" = hanya data/train.jsonl, "mix" = data sendiri + dataset umum
REPO_URL      = "https://github.com/PvzKing/Dataset.git"
BRANCH        = "main"          # sebelum PR digabung ke main: "claude/wizardly-knuth-htykil"
MAX_SEQ_LEN   = 8192            # sampel yang lebih panjang DIBUANG, tidak dipotong, supaya kode HTML tidak terpenggal
EPOCHS        = 2 if DATASET == "mix" else 3
LEARNING_RATE = 1e-4
LORA_R, LORA_ALPHA      = 16, 32
BATCH_SIZE, GRAD_ACCUM  = 1, 8  # batch efektif 8
SAVE_STEPS    = 10              # simpan checkpoint ke Drive setiap N step
SEED          = 42
DRIVE_ROOT    = "/content/drive/MyDrive/finetune-id"
EXPORT_GGUF   = False           # True: buat file GGUF q4_k_m untuk Ollama / llama.cpp

MODELS = {
    # Salinan Unsloth dari Qwen/Qwen3.5-4B. LoRA 16-bit, karena Unsloth tidak menyarankan QLoRA 4-bit untuk Qwen3.5.
    "qwen3.5-4b": dict(name="unsloth/Qwen3.5-4B", load_in_4bit=False, template_kwargs={"enable_thinking": False}),
    # Qwen/Qwen2.5-Coder-7B-Instruct yang sudah dikuantisasi 4-bit (QLoRA), supaya muat di T4.
    "qwen2.5-coder-7b": dict(name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit", load_in_4bit=True, template_kwargs={}),
}
CFG = MODELS[MODEL]
TEMPLATE_KWARGS = CFG["template_kwargs"]
DRIVE_DIR = f"{DRIVE_ROOT}/{MODEL}-{DATASET}"

## 3. Hubungkan Google Drive dan ambil repo
Jika repo **privat**, buat *fine-grained token* GitHub (akses baca ke repo ini), simpan di panel 🔑 *Secrets*
Colab dengan nama `GITHUB_TOKEN`, dan aktifkan akses notebook ke secret itu.

In [ ]:
import os
import subprocess
from google.colab import drive

drive.mount("/content/drive")
CKPT_DIR = f"{DRIVE_DIR}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)


def sh(*cmd):
    """Jalankan perintah dan hentikan notebook jika gagal."""
    if subprocess.run(cmd).returncode != 0:
        raise RuntimeError(f"perintah gagal: {cmd[0]} {cmd[1] if len(cmd) > 1 else ''}")


if not os.path.isdir("/content/Dataset"):
    clone_url = REPO_URL
    try:
        from google.colab import userdata
        clone_url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass  # tanpa token: repo publik
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, clone_url, "/content/Dataset")
    sh("git", "-C", "/content/Dataset", "remote", "set-url", "origin", REPO_URL)  # token tidak disimpan di .git/config
os.chdir("/content/Dataset")
sh("git", "log", "-1", "--format=commit %h  %s")

## 4. Siapkan file data
Untuk `mix`, dataset umum diunduh dari Hugging Face lalu disimpan ke Drive. Saat sesi diulang, file yang sama dipakai
lagi supaya urutan data tetap identik dan checkpoint bisa dilanjutkan dengan benar.

In [ ]:
import shutil

if DATASET == "mix":
    mix_on_drive = f"{DRIVE_ROOT}/train_mix.jsonl"
    if os.path.exists(mix_on_drive):
        shutil.copy(mix_on_drive, "data/train_mix.jsonl")
        print("memakai train_mix.jsonl yang sudah ada di Drive")
    else:
        sh("python", "scripts/mix_general.py", "--seed", str(SEED))
        shutil.copy("data/train_mix.jsonl", mix_on_drive)
    DATA_FILE = "data/train_mix.jsonl"
else:
    DATA_FILE = "data/train.jsonl"

sh("python", "scripts/validate.py", DATA_FILE)

## 5. Muat model dan pasang LoRA
`unsloth` harus di-import sebelum `transformers` dan `trl`. Presisi dipilih otomatis sesuai GPU. Untuk Qwen3.5,
Unsloth hanya memuat bagian teks model, tanpa vision encoder, supaya hemat VRAM.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["name"],
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=CFG["load_in_4bit"],
    load_in_16bit=not CFG["load_in_4bit"],
)
# Model multimodal bisa mengembalikan processor; training teks cukup memakai tokenizer di dalamnya.
if hasattr(tokenizer, "tokenizer"):
    processor, tokenizer = tokenizer, tokenizer.tokenizer
    if not getattr(tokenizer, "chat_template", None):
        tokenizer.chat_template = processor.chat_template

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,                  # 0 adalah jalur tercepat di Unsloth
    bias="none",
    target_modules=None,             # Unsloth memilih semua layer linear di attention (termasuk Gated DeltaNet) dan MLP
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    use_gradient_checkpointing="unsloth",  # menghemat VRAM untuk konteks panjang
    random_state=SEED,
)

## 6. Siapkan data
Setiap percakapan diubah ke chat template bawaan model. Jawaban terakhir ditempel tepat setelah prompt inferensi
(`add_generation_prompt=True`), sehingga format training sama persis dengan saat model dipakai, termasuk blok
`<think>` kosong pada mode non-thinking Qwen3.5.

In [ ]:
import json
import math
from datasets import Dataset

EOS = "<|im_end|>"


def render(messages, **kw):
    return tokenizer.apply_chat_template(messages, tokenize=False, **TEMPLATE_KWARGS, **kw)


def to_text(messages):
    *history, last = messages
    assert last["role"] == "assistant"
    return render(history, add_generation_prompt=True) + last["content"] + EOS


probe = [{"role": "user", "content": "halo"}]
print("prompt jawaban saat inferensi:", repr(render(probe, add_generation_prompt=True).removeprefix(render(probe))))

rows = [json.loads(line) for line in open(DATA_FILE, encoding="utf-8")]
texts, lengths, dropped = [], [], []
for row in rows:
    text = to_text(row["messages"])
    n = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    if n > MAX_SEQ_LEN:
        dropped.append((row["id"], n))
        continue
    texts.append(text)
    lengths.append(n)

train_dataset = Dataset.from_dict({"text": texts})
total_steps = math.ceil(len(texts) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
trained_tokens = sum(lengths) * EPOCHS

print(f"{len(texts)} sampel dipakai, {len(dropped)} dibuang karena > {MAX_SEQ_LEN} token {dropped[:5]}")
print(f"token per epoch: {sum(lengths):,}  |  terpanjang: {max(lengths):,}  |  rata-rata: {sum(lengths) // len(lengths):,}")
print(f"{EPOCHS} epoch = {total_steps} step, ±{trained_tokens:,} token dilatih")
lo, hi = (300, 900) if is_bfloat16_supported() else (60, 200)  # token/detik, perkiraan kasar
print(f"perkiraan waktu ({lo}–{hi} token/detik): {trained_tokens / hi / 60:.0f}–{trained_tokens / lo / 60:.0f} menit")
print("\ncontoh awal teks training:\n" + texts[0][:400])

## 7. Trainer
Loss hanya dihitung pada jawaban assistant (`train_on_responses_only`), jadi model belajar menjawab, bukan menghafal pertanyaan.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir=CKPT_DIR,
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
        packing=False,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, round(0.1 * total_steps)),  # 10% dari total step
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),  # Unsloth mengganti ke float32 sendiri jika model tidak aman di fp16
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        seed=SEED,
        report_to="none",
        dataset_num_proc=2,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Cek masking: bagian yang ikut dilatih harus diawali jawaban assistant, bukan system prompt atau pertanyaan.
labels = trainer.train_dataset[0]["labels"]
trained = tokenizer.decode([t for t in labels if t != -100])
print(f"{sum(t != -100 for t in labels)} dari {len(labels)} token sampel pertama ikut dilatih:")
print(trained[:300], "...")

## 8. Training
Jika ada checkpoint di Drive dari sesi sebelumnya, training dilanjutkan dari situ.
Jika muncul *CUDA out of memory*, turunkan `MAX_SEQ_LEN` ke 6144 di sel Konfigurasi lalu jalankan ulang semua sel.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_ckpt = get_last_checkpoint(CKPT_DIR)
print("melanjutkan dari", last_ckpt if last_ckpt else "awal")
stats = trainer.train(resume_from_checkpoint=last_ckpt)
print(f"selesai dalam {stats.metrics['train_runtime'] / 60:.0f} menit, loss akhir {stats.metrics['train_loss']:.3f}")

## 9. Simpan adapter LoRA ke Drive
Adapter ini dipasang di atas model dasarnya (`Qwen/Qwen3.5-4B` atau `Qwen/Qwen2.5-Coder-7B-Instruct`).

In [ ]:
ADAPTER_DIR = f"{DRIVE_DIR}/lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter disimpan di", ADAPTER_DIR)

## 10. Coba hasilnya
Prompt di bawah **tidak ada** di dataset. Hasilnya disimpan ke Drive sebagai file HTML. Jalankan notebook ini dengan
`MODEL` yang berbeda untuk membandingkan model dengan prompt yang sama. Di T4, menghasilkan ±4.000 token butuh beberapa menit.

In [ ]:
import html
from IPython.display import HTML, display
from transformers import TextStreamer

FastLanguageModel.for_inference(model)
TEST_PROMPTS = {
    "bengkel": "Buatkan landing page satu file HTML untuk bengkel motor \"Gaspol Motor\" di Yogyakarta: hero dengan "
               "tombol booking servis, daftar layanan dan harga, testimoni, jam buka, dan form booking dengan validasi. "
               "Responsif dan tanpa library eksternal.",
    "absensi": "Buat aplikasi absensi kelas dalam satu file HTML: tambah siswa, tandai hadir/izin/sakit/alfa per tanggal, "
               "rekap persentase kehadiran, simpan di localStorage, dan tombol ekspor CSV.",
}

for key, prompt in TEST_PROMPTS.items():
    inputs = tokenizer(render([{"role": "user", "content": prompt}], add_generation_prompt=True),
                       return_tensors="pt", add_special_tokens=False).to("cuda")
    output = model.generate(
        **inputs, max_new_tokens=6000, temperature=0.7, top_p=0.8, top_k=20, repetition_penalty=1.05,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    if "```html" not in answer:
        print(f"[{key}] tidak ada blok ```html di jawaban")
        continue
    page = answer.split("```html", 1)[1].split("```", 1)[0]
    with open(f"{DRIVE_DIR}/contoh-{key}.html", "w", encoding="utf-8") as f:
        f.write(page)
    display(HTML(f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:600px;border:1px solid #ccc"></iframe>'))

## 11. (Opsional) Ekspor GGUF untuk Ollama / llama.cpp
Set `EXPORT_GGUF = True` di sel konfigurasi. Proses ini menggabungkan adapter ke model dasar lalu mengkuantisasi ke
q4_k_m (±2,5–3 GB untuk 4B). Butuh ±15–40 menit.

In [ ]:
if EXPORT_GGUF:
    model.save_pretrained_gguf(f"{DRIVE_DIR}/gguf", tokenizer, quantization_method="q4_k_m")
    print("GGUF disimpan di", f"{DRIVE_DIR}/gguf")
else:
    print("dilewati (EXPORT_GGUF = False)")